## Vendor Effects Figure: Shared Setup


In [ ]:
# --- Setup: libraries ---
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(forcats)
  library(fs)
  library(jsonlite)
  library(purrr)
  library(scales)
  library(viridis)
  library(arrow)
  library(patchwork)
})

# --- Config ---
if (identical(Sys.getenv("CONFIG_PATH", unset = ""), "")) {
  Sys.setenv(CONFIG_PATH = "/mnt/isilon/bgdlab_hbcd/projects/meisler_abcd_dmri/config.json")
  message("[INFO] CONFIG_PATH was not set. Using default project config.json")
}

config_path <- Sys.getenv("CONFIG_PATH", unset = "")
if (!nzchar(config_path)) stop("CONFIG_PATH is not set")
if (!file_exists(config_path)) stop("CONFIG_PATH not found: ", config_path)

config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, mustWork = FALSE)

plot_style_file <- fs::path(project_root, "scripts", "utils", "plot_style.R")
if (!file_exists(plot_style_file)) {
  stop("Missing plot style helper: ", plot_style_file)
}
source(plot_style_file)

# --- Theme ---
plot_style <- get_plot_style(config)

theme_pub <- make_theme_pub(
  style = plot_style,
  legend_position = "right",
  axis_title_pt = 18,
  axis_text_pt = 16,
  plot_title_pt = 20,
  legend_title_pt = 16,
  legend_text_pt = 14,
  base_size_pt = 11
)

# --- Options and metric/bundle labels ---
qc_target <- "no_quality"
metrics_keep <- c("DKI_mkt", "NODDI_icvf", "MAPMRI_rtop", "GQI_fa", "GQI_md")

metric_labels <- c(
  "DKI_mkt" = "MKT",
  "NODDI_icvf" = "ICVF",
  "MAPMRI_rtop" = "RTOP",
  "GQI_fa" = "FA",
  "GQI_md" = "MD"
)

# same metric palette used previously
metric_colors <- c(
  "MKT" = "#1f77b4",
  "ICVF" = "#2ca02c",
  "RTOP" = "#ff7f0e",
  "FA" = "#d62728",
  "MD" = "#9467bd"
)

category_order <- c(
  "Association",
  "Projection (Basal Ganglia)",
  "Projection (Brainstem)",
  "Cerebellum",
  "Commissure"
)

# --- Data paths and load vendor effects ---
assembled_vendor_files <- c(
  fs::path(project_root, "data", "vendor_effects", "vendor_effects_all_outputs.rds"),
  fs::path(project_root, "data", "vendor_effects", "vendor_effects_pairwise_all_outputs.rds")
)
assembled_vendor_file <- assembled_vendor_files[file_exists(assembled_vendor_files)][1]

vendor_dirs <- c(
  fs::path(project_root, "data", "vendor_effects", "vendor_effects_outputs"),
  fs::path(project_root, "data", "vendor_effects", "vendor_effects_outputs_pairwise")
)
vendor_dir <- vendor_dirs[dir_exists(vendor_dirs)][1]

if (!is.na(assembled_vendor_file) && nzchar(assembled_vendor_file)) {
  df_vendor_all <- readRDS(assembled_vendor_file)
  if (!is.data.frame(df_vendor_all)) {
    stop("Assembled vendor effects file is not a data.frame: ", assembled_vendor_file)
  }
} else {
  message("[WARN] Assembled vendor effects file not found. Falling back to file scan.")
  if (is.na(vendor_dir) || !nzchar(vendor_dir)) {
    stop("Vendor effects directory not found. Checked: ", paste(vendor_dirs, collapse = ", "))
  }
  vendor_files <- list.files(vendor_dir, pattern = "_vendor_effects\\.rds$", full.names = TRUE)
  if (length(vendor_files) == 0) {
    stop("No _vendor_effects.rds files found in ", vendor_dir)
  }
  df_vendor_all <- purrr::map_dfr(vendor_files, readRDS)
}

if (!all(c("source", "qc_covariate", "metric", "bundle", "bundle_category", "effect_size") %in% names(df_vendor_all))) {
  stop("Vendor assembled data missing required columns")
}

options(repr.plot.width = 20, repr.plot.height = 12, repr.plot.res = 120)



## Panel 1 (Top): Figure6-Style Raw Vendor-Effect Heatmap


In [ ]:
# Figure6-like heatmap for raw vendor effects

df_heat <- df_vendor_all %>%
  filter(
    source == "raw",
    qc_covariate == qc_target,
    metric %in% metrics_keep,
    !is.na(bundle), !is.na(bundle_category), !is.na(effect_size)
  ) %>%
  mutate(
    metric_label = unname(metric_labels[metric]),
    bundle_clean = bundle,
    bundle_category = case_when(
      bundle_category == "ProjectionBasalGanglia" ~ "Projection (Basal Ganglia)",
      bundle_category == "ProjectionBrainstem" ~ "Projection (Brainstem)",
      TRUE ~ bundle_category
    ),
    bundle_clean = bundle_clean %>%
      str_replace("FrontoOccipital", "Fronto-Occipital") %>%
      str_replace("NonDecussating", "Non-Decussating") %>%
      str_replace("L$", " (L)") %>%
      str_replace("R$", " (R)") %>%
      str_replace_all("(?<=[a-z])(?=[A-Z])", " ")
  )

if (nrow(df_heat) == 0) stop("No rows for Panel 1 (raw + no_quality + metrics_keep)")

metric_order <- df_heat %>%
  group_by(metric_label) %>%
  summarise(mean_effect = mean(effect_size, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(mean_effect)) %>%
  pull(metric_label)

df_heat$metric_label <- factor(df_heat$metric_label, levels = metric_order)

bundle_df <- df_heat %>%
  group_by(bundle_category, bundle_clean) %>%
  summarise(mean_effect = mean(effect_size, na.rm = TRUE), .groups = "drop") %>%
  mutate(bundle_category = factor(bundle_category, levels = category_order)) %>%
  arrange(bundle_category, desc(mean_effect), bundle_clean)

bundle_levels <- c()
for (cat in category_order) {
  cat_bundles <- bundle_df %>% filter(bundle_category == cat) %>% pull(bundle_clean)
  if (length(cat_bundles) > 0) {
    bundle_levels <- c(bundle_levels, cat_bundles, paste0("spacer_", cat))
  }
}
bundle_levels <- bundle_levels[!grepl("^spacer_Commissure$", bundle_levels)]

df_heat$bundle_clean <- factor(df_heat$bundle_clean, levels = bundle_levels)

spacer_df <- expand.grid(
  bundle_clean = bundle_levels[grepl("^spacer_", bundle_levels)],
  metric_label = metric_order
) %>% mutate(effect_size = NA_real_)

df_heat_plot <- bind_rows(
  df_heat %>% select(bundle_clean, metric_label, effect_size),
  spacer_df
)

x_labels <- levels(df_heat_plot$bundle_clean)
x_labels[grepl("^spacer_", x_labels)] <- ""

bundle_index_df <- tibble(bundle_clean = bundle_levels, index = seq_along(bundle_levels))
cat_positions <- bundle_df %>%
  group_by(bundle_category) %>%
  summarise(
    xmin = min(bundle_index_df$index[bundle_index_df$bundle_clean %in% bundle_clean]),
    xmax = max(bundle_index_df$index[bundle_index_df$bundle_clean %in% bundle_clean]),
    xmid = mean(c(xmin, xmax)),
    .groups = "drop"
  ) %>%
  filter(bundle_category != "Commissure")

heatmap_export_width_in <- 20
heatmap_final_width_in <- 20

p_heatmap <- ggplot(df_heat_plot, aes(x = bundle_clean, y = metric_label, fill = effect_size)) +
  geom_tile(color = "white", linewidth = 0.5) +
  scale_fill_viridis_c(
    option = "mako",   # different from age heatmap colormap
    name = expression("Vendor effect (" * Delta * R^2 * ")"),
    na.value = "white"
  ) +
  scale_x_discrete(labels = x_labels, expand = expansion(add = 0.8)) +
  scale_y_discrete(expand = expansion(mult = c(0.01, 0.05))) +
  coord_cartesian(clip = "off") +
  labs(x = NULL, y = NULL) +
  theme_minimal(
    base_family = plot_style$font_family,
    base_size = pt_for_export(11, heatmap_export_width_in, heatmap_final_width_in)
  ) +
  theme(
    panel.grid = element_blank(),
    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      vjust = 1,
      size = pt_for_export(10, heatmap_export_width_in, heatmap_final_width_in)
    ),
    axis.text.y = element_text(size = pt_for_export(16, heatmap_export_width_in, heatmap_final_width_in)),
    axis.ticks = element_blank(),
    plot.margin = margin(70, 20, 40, 100),
    legend.title = element_text(size = pt_for_export(16, heatmap_export_width_in, heatmap_final_width_in)),
    legend.text = element_text(size = pt_for_export(11, heatmap_export_width_in, heatmap_final_width_in)),
    panel.background = element_rect(fill = plot_style$panel_background_fill, color = NA),
    plot.background = element_rect(fill = plot_style$plot_background_fill, color = NA),
    panel.border = element_blank()
  )

y_top <- length(metric_order) + 0.85
for (i in seq_len(nrow(cat_positions))) {
  p_heatmap <- p_heatmap + annotate(
    "text",
    x = cat_positions$xmid[i],
    y = y_top,
    label = cat_positions$bundle_category[i],
    family = plot_style$font_family,
    size = pt_for_export(16, heatmap_export_width_in, heatmap_final_width_in) / .pt
  )
}

print(p_heatmap)



## Panel 2 (Bottom Left): Mean Vendor Effect Across Bundles (Raw vs Harmonized)


In [ ]:
df_line <- df_vendor_all %>%
  filter(
    source %in% c("raw", "harmonized"),
    qc_covariate == qc_target,
    metric %in% metrics_keep,
    !is.na(bundle),
    !is.na(effect_size)
  ) %>%
  group_by(source, metric, bundle) %>%
  summarise(effect_size = mean(effect_size, na.rm = TRUE), .groups = "drop") %>%
  group_by(source, metric) %>%
  summarise(
    mean_effect = mean(effect_size, na.rm = TRUE),
    n_bundle = dplyr::n(),
    sem_effect = if_else(n_bundle > 1, sd(effect_size, na.rm = TRUE) / sqrt(n_bundle), as.numeric(NA)),
    .groups = "drop"
  ) %>%
  mutate(
    source = factor(source, levels = c("raw", "harmonized"), labels = c("Raw", "Harmonized")),
    metric_label = factor(recode(metric, !!!metric_labels), levels = unname(metric_labels))
  )

if (nrow(df_line) == 0) stop("No rows for Panel 2")

p_line <- ggplot(df_line, aes(x = source, y = mean_effect, group = metric_label, color = metric_label)) +
  geom_line(linewidth = 1.3) +
  geom_point(size = 3.0) +
  geom_errorbar(aes(ymin = mean_effect - sem_effect, ymax = mean_effect + sem_effect), width = 0.06, alpha = 0.85) +
  scale_color_manual(values = metric_colors, name = "Metric") +
  labs(
    title = "Mean Vendor Effect Across Bundles",
    subtitle = paste0("QC setting: ", qc_target),
    x = NULL,
    y = expression("Vendor effect (" * Delta * R^2 * ")")
  ) +
  theme_pub +
  theme(
    panel.grid = element_blank(),
    legend.position = "right",
    legend.title = element_text(face = "bold", size = 13),
    legend.text = element_text(size = 11),
    panel.background = element_rect(fill = plot_style$panel_background_fill, color = NA),
    plot.background = element_rect(fill = plot_style$plot_background_fill, color = NA),
    panel.border = element_blank()
  )

print(p_line)



## Panel 3 (Bottom Right): Representative Vendor Distributions (Before/After Harmonization)


In [ ]:
# Representative distribution plot:
# metric = GQI_fa, bundle = Cerebellum_InferiorCerebellarPeduncleR

raw_path <- fs::path(project_root, "data", "raw_data", "merged_data_meisler_analyses.parquet")
harm_path <- fs::path(project_root, "data", "harmonized_data", "merged_data_meisler_analyses_harmonized.parquet")
if (!file_exists(raw_path)) stop("Missing raw parquet: ", raw_path)
if (!file_exists(harm_path)) stop("Missing harmonized parquet: ", harm_path)

metric_rep <- "GQI_fa"
bundle_cat_rep <- "Cerebellum"
bundle_rep <- "InferiorCerebellarPeduncleR"
bundle_col <- paste0("bundle_", bundle_cat_rep, "_", bundle_rep, "_", metric_rep, "_median")

raw_df <- arrow::read_parquet(raw_path)
harm_df <- arrow::read_parquet(harm_path)

if (!all(c("scanner_manufacturer", bundle_col) %in% names(raw_df))) {
  stop("Raw data missing scanner_manufacturer or bundle column: ", bundle_col)
}
if (!all(c("scanner_manufacturer", bundle_col) %in% names(harm_df))) {
  stop("Harmonized data missing scanner_manufacturer or bundle column: ", bundle_col)
}

vendor_short <- function(x) {
  case_when(
    str_detect(str_to_lower(as.character(x)), "siemens") ~ "Siemens",
    str_detect(str_to_lower(as.character(x)), "philips") ~ "Philips",
    str_detect(str_to_lower(as.character(x)), "ge") ~ "GE",
    TRUE ~ as.character(x)
  )
}

df_rep <- bind_rows(
  raw_df %>%
    transmute(source = "Raw", scanner_manufacturer, value = as.numeric(.data[[bundle_col]])),
  harm_df %>%
    transmute(source = "Harmonized", scanner_manufacturer, value = as.numeric(.data[[bundle_col]]))
) %>%
  mutate(vendor = vendor_short(scanner_manufacturer)) %>%
  filter(!is.na(vendor), !is.na(value)) %>%
  mutate(source = factor(source, levels = c("Raw", "Harmonized")))

if (nrow(df_rep) == 0) stop("No rows for Panel 3 representative plot")

# Global vendor-difference significance (within each source) via Kruskal-Wallis
star_p <- function(p) {
  dplyr::case_when(
    is.na(p) ~ "n.s.",
    p < 0.001 ~ "***",
    p < 0.01 ~ "**",
    p < 0.05 ~ "*",
    TRUE ~ "n.s."
  )
}

sig_df <- df_rep %>%
  group_by(source) %>%
  summarise(
    p_kw = tryCatch(stats::kruskal.test(value ~ vendor)$p.value, error = function(e) as.numeric(NA)),
    y_pos = quantile(value, 0.98, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(label = paste0("Kruskal-Wallis ", star_p(p_kw), " (p=", scales::pvalue(p_kw, accuracy = 0.001), ")"))

p_rep <- ggplot(df_rep, aes(x = vendor, y = value, fill = vendor)) +
  geom_violin(alpha = 0.45, color = NA, trim = TRUE) +
  geom_boxplot(width = 0.16, outlier.shape = NA, alpha = 0.75, color = "black") +
  geom_jitter(width = 0.08, alpha = 0.12, size = 0.6, color = "black") +
  facet_wrap(~source, nrow = 1, scales = "free_y") +
  geom_text(
    data = sig_df,
    aes(x = 2, y = y_pos, label = label),
    inherit.aes = FALSE,
    vjust = -0.4,
    size = 4.0
  ) +
  scale_fill_brewer(palette = "Set2", guide = "none") +
  labs(
    title = "Representative Vendor Distributions",
    subtitle = paste0(metric_rep, " | ", bundle_cat_rep, "_", bundle_rep),
    x = NULL,
    y = metric_labels[[metric_rep]]
  ) +
  theme_pub +
  theme(
    panel.grid = element_blank(),
    strip.text = element_text(face = "bold", size = 12),
    panel.background = element_rect(fill = plot_style$panel_background_fill, color = NA),
    plot.background = element_rect(fill = plot_style$plot_background_fill, color = NA),
    panel.border = element_blank()
  )

print(p_rep)



## Combined Figure Layout (Top Full-Width + Two Bottom Panels)


In [ ]:
# Compose final figure:
# top row: heatmap full width
# bottom row: line plot (left) + representative distribution (right)

combined_fig <-
  p_heatmap /
  (p_line | p_rep) +
  plot_layout(heights = c(1.35, 1.0), widths = c(1, 1))

print(combined_fig)

